## 📝 Chap07-1. Function Calling Basics

#### 펑션 콜링이란?
- 펑션 콜링(Function Calling)은 대규모 언어 모델(LLM)이 직접 해결할 수 없는 문제를 풀기 위해 외부의 프로그램이나 API(함수)를 선택하고, 이에 필요한 인자(Arguments)를 구조화된 JSON 데이터로 만들어주는 기능입니다.
- 개발자는 LLM이 함수를 언제, 어떻게 써야 하는지 알 수 있도록 JSON 스키마 형태로 도구를 정의하여 모델에 주입해야 합니다.

In [6]:
from gpt_functions import get_current_time, tools 
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")  # 환경 변수에서 API 키 가져오기

client = OpenAI(api_key=api_key)  # 오픈AI 클라이언트의 인스턴스 생성

def get_ai_response(messages, tools=None):
    response = client.chat.completions.create(
        model="gpt-4o",  # 응답 생성에 사용할 모델 지정
        messages=messages,  # 대화 기록을 입력으로 전달
        tools=tools,  # 사용 가능한 도구 목록 전달
    )
    return response  # 생성된 응답 내용 반환



messages = [
    {"role": "system", "content": "너는 사용자를 도와주는 상담사야."},  # 초기 시스템 메시지
]

while True:
    user_input = input("사용자\t: ")  # 사용자 입력 받기

    if user_input == "exit":  # 사용자가 대화를 종료하려는지 확인
        break
    
    messages.append({"role": "user", "content": user_input})  # 사용자 메시지 대화 기록에 추가
    
    ai_response = get_ai_response(messages, tools=tools)
    ai_message = ai_response.choices[0].message
    print(ai_message)  # ③ gpt에서 반환되는 값을 파악하기 위해 임시로 추가

    tool_calls = ai_message.tool_calls  # AI 응답에 포함된 tool_calls를 가져옵니다.
    if tool_calls:  # tool_calls가 있는 경우
        for tool_call in tool_calls:
            tool_name = tool_call.function.name # 실행해야한다고 판단한 함수명 받기
            tool_call_id = tool_call.id         # tool_call 아이디 받기    
            arguments = json.loads(tool_call.function.arguments) # (1) 문자열을 딕셔너리로 변환    
            
            if tool_name == "get_current_time":  # ⑤ 만약 tool_name이 "get_current_time"이라면
                messages.append({
                    "role": "function",  # role을 "function"으로 설정
                    "tool_call_id": tool_call_id,
                    "name": tool_name,
                    "content": get_current_time(timezone=arguments['timezone']),  # 타임존 추가
                })
        messages.append({"role": "system", "content": "이제 주어진 결과를 바탕으로 답변할 차례다."})  # 함수 실행 완료 메시지 추가
        ai_response = get_ai_response(messages, tools=tools) # 다시 GPT 응답 받기
        ai_message = ai_response.choices[0].message

    messages.append(ai_message)  # AI 응답을 대화 기록에 추가하기

    print("AI\t: " + ai_message.content)  # AI 응답 출력

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_FaZzVBljttAYtQzfr5UIKqXM', function=Function(arguments='{"timezone": "America/New_York"}', name='get_current_time'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_EinYE1fJpY1DQc5HzMZt8kKL', function=Function(arguments='{"timezone": "Europe/London"}', name='get_current_time'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_kP0Q5a5bFOcPNkNIp1tNkE7r', function=Function(arguments='{"timezone": "Europe/Paris"}', name='get_current_time'), type='function')])
2026-08-29 02:45:31 America/New_York
2026-08-29 07:45:31 Europe/London
2026-08-29 08:45:31 Europe/Paris
AI	: 현재 시각은 다음과 같습니다:

- 뉴욕: 2026년 8월 29일, 오전 2시 45분
- 런던: 2026년 8월 29일, 오전 7시 45분
- 파리: 2026년 8월 29일, 오전 8시 45분


#### 스트림릿에서 펑션 콜링 사용하기

In [ ]:
streamlit run what_time_is_it_streamlit.py